# Modelos Transformer (DistilBERT y RoBERTa-BNE)

Clasificación binaria por patología (4 clasificadores independientes) sobre reportes radiológicos en español, usando fine-tuning de transformers preentrenados.

**Pipeline:**
1. `distilbert-base-multilingual-cased` — primero (más rápido, ~2-3h CPU).
2. `PlanTL-GOB-ES/roberta-base-bne` — segundo (más pesado, si hay tiempo).

**Datos:**
- `train_augmentado_llm.xlsx` — Train (real + sintético LLM) y Test (solo reales, 143).
- `train_oversampling_data.xlsx` (`Train_ROS`) — boost adicional para ACV.

**Patologías:** `acv`, `hemorragia`, `desviacion_linea_media`, `fractura_compleja_craneo`.

In [2]:
import ctypes
# Evita que Windows suspenda el PC mientras corre el código
ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)
print("Modo 'no suspender' activado")

Modo 'no suspender' activado


In [3]:
# Celda 0 — instalación
%pip install transformers torch scikit-learn openpyxl tqdm --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Celda 1 — configuración global
import os, time, json, warnings, logging
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Silenciar deprecaciones de transformers
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
logging.getLogger('transformers').setLevel(logging.ERROR)

# ---------- Rutas ----------
posibles_rutas_base = [Path('../data'), Path('data'), Path.cwd() / 'data', Path.cwd().parent / 'data']
DATA_DIR = next((p.resolve() for p in posibles_rutas_base if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("No se encontró el directorio 'data'.")

LLM_PATH = DATA_DIR / 'train_augmentado_llm.xlsx'
ROS_PATH = DATA_DIR / 'train_oversampling_data.xlsx'

MODELS_DIR = (DATA_DIR.parent / 'models').resolve()
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Modelos a entrenar ----------
MODELOS = {
    'distilbert': {
        'hf_name': 'distilbert-base-multilingual-cased',
        'short':   'distilbert',
        'epochs':  3,
    },
    'roberta_bne': {
        'hf_name': 'PlanTL-GOB-ES/roberta-base-bne',
        'short':   'roberta_bne',
        'epochs':  4,
    }
}

# ---------- Hiperparámetros comunes ----------
MAX_LENGTH    = 256
BATCH_SIZE    = 8       # CPU-friendly
LEARNING_RATE = 2e-5
RANDOM_SEED   = 42
PATOLOGIAS    = ['acv', 'hemorragia', 'desviacion_linea_media', 'fractura_compleja_craneo']

ETIQUETAS = {
    'hemorragia':                ['Sin hemorragia', 'Hemorragia'],
    'acv':                       ['Sin ACV', 'ACV'],
    'fractura_compleja_craneo':  ['Sin fractura compleja de craneo', 'Fractura compleja de craneo'],
    'desviacion_linea_media':    ['Sin desviacion de linea media', 'Desviacion de linea media']
}

# ACV usa el dataset combinado LLM + ROS
USA_ROS_BOOST = {'acv'}

import torch
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"DATA_DIR:   {DATA_DIR}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"Device:     {DEVICE}")
print(f"Torch:      {torch.__version__}")
print()
print(f"max_length = {MAX_LENGTH} | batch = {BATCH_SIZE} | lr = {LEARNING_RATE}")
print()
print("Estimación de tiempo (CPU, 4 patologías):")
print(f"  - DistilBERT  (~3 epochs):  ~2-3 horas")
print(f"  - RoBERTa-BNE (~4 epochs):  ~4-6 horas")
print("  (en GPU tiempos ~10-20x menores)")

DATA_DIR:   C:\Users\Isabella\Documents\ICESI\8. Octavo Semestre\Proyecto de Grado I\PDG-Diagnostico-Clinico\data
MODELS_DIR: C:\Users\Isabella\Documents\ICESI\8. Octavo Semestre\Proyecto de Grado I\PDG-Diagnostico-Clinico\models
Device:     cpu
Torch:      2.11.0+cpu

max_length = 256 | batch = 8 | lr = 2e-05

Estimación de tiempo (CPU, 4 patologías):
  - DistilBERT  (~3 epochs):  ~2-3 horas
  - RoBERTa-BNE (~4 epochs):  ~4-6 horas
  (en GPU tiempos ~10-20x menores)


## DistilBERT con datos críticos (rad_criticos.xlsx)

**Objetivo:** Fine-tuning de DistilBERT exclusivamente sobre los reportes críticos sin augmentación, para establecer la línea base a comparar contra las estrategias LLM y ROS.

In [ ]:
# =============================================================================
# CARGA DE DATOS CRÍTICOS (rad_criticos.xlsx) — sin augmentación
# Entrenamiento DistilBERT en celda posterior a la definición de entrenar_todas.
# =============================================================================
from sklearn.model_selection import train_test_split as _tts

_crit_path = DATA_DIR / 'rad_criticos.xlsx'
if not _crit_path.exists():
    raise FileNotFoundError(f'No se encontró {_crit_path}')

df_criticos_raw = pd.read_excel(str(_crit_path))
for _col in ['Hallazgos', 'Opinión']:
    df_criticos_raw[_col] = df_criticos_raw[_col].fillna('').astype(str)
df_criticos_raw['texto'] = (
    df_criticos_raw['Hallazgos'] + ' ' + df_criticos_raw['Opinión']).str.strip()

df_crit_train, df_crit_test = _tts(
    df_criticos_raw, test_size=0.2, random_state=RANDOM_SEED,
    stratify=df_criticos_raw['hemorragia'])

print(f'Críticos — Train: {len(df_crit_train)} | Test: {len(df_crit_test)}')
print(f'\n{"Patología":<35}  {"Train+":>6}  {"Test+":>6}')
print('-' * 55)
for p in PATOLOGIAS:
    n_tr = int(df_crit_train[p].sum())
    n_te = int(df_crit_test[p].sum())
    print(f'  {p:<33}  {n_tr:>6}  {n_te:>6}')
print('\n(DistilBERT con datos críticos se entrena'
      ' después de definir entrenar_todas.)')


In [5]:
# Celda 2 — carga de datos
if not LLM_PATH.exists():
    raise FileNotFoundError(f"No se encontró '{LLM_PATH}'.")

df_train = pd.read_excel(LLM_PATH, sheet_name='Train')
df_test  = pd.read_excel(LLM_PATH, sheet_name='Test')

for d in (df_train, df_test):
    d['Hallazgos'] = d['Hallazgos'].fillna('').astype(str)
    d['Opinión']   = d['Opinión'].fillna('').astype(str)
    d['texto']     = (d['Hallazgos'] + ' ' + d['Opinión']).str.strip()

# Boost ACV: Train + Train_ROS
if ROS_PATH.exists():
    df_ros = pd.read_excel(ROS_PATH, sheet_name='Train_ROS')
    df_ros['Hallazgos'] = df_ros['Hallazgos'].fillna('').astype(str)
    df_ros['Opinión']   = df_ros['Opinión'].fillna('').astype(str)
    df_ros['texto']     = (df_ros['Hallazgos'] + ' ' + df_ros['Opinión']).str.strip()
    df_train_acv = pd.concat([df_train, df_ros], ignore_index=True)
    print(f"Train_ROS cargado: {len(df_ros)} registros")
else:
    df_train_acv = df_train.copy()
    print("Train_ROS no disponible — ACV usa solo datos LLM.")

print(f"Train LLM:       {len(df_train)} registros")
print(f"Train ACV boost: {len(df_train_acv)} registros (LLM + ROS)")
print(f"Test:            {len(df_test)} registros (solo reales)")
print()
print(f"{'Patología':<35s}  {'Train+':>6}  {'Test+':>6}")
print('-' * 55)
for p in PATOLOGIAS:
    n_tr = int(df_train[p].sum())
    n_te = int(df_test[p].sum())
    print(f"  {p:<33s}  {n_tr:>6}  {n_te:>6}")

print()
print(f"Longitud media (palabras) Train: {df_train['texto'].str.split().str.len().mean():.0f}")
print(f"Longitud media (palabras) Test : {df_test['texto'].str.split().str.len().mean():.0f}")

Train_ROS cargado: 761 registros
Train LLM:       1334 registros
Train ACV boost: 2095 registros (LLM + ROS)
Test:            143 registros (solo reales)

Patología                            Train+   Test+
-------------------------------------------------------
  acv                                   277      46
  hemorragia                            687     126
  desviacion_linea_media                332      22
  fractura_compleja_craneo              325       5

Longitud media (palabras) Train: 192
Longitud media (palabras) Test : 173


In [6]:
# Celda 3 — función entrenar_clasificador
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, precision_recall_curve,
    roc_auc_score, cohen_kappa_score, confusion_matrix
)
from tqdm.auto import tqdm


class TextDataset(Dataset):
    def __init__(self, textos, etiquetas, tokenizer, max_length):
        self.textos = list(textos)
        self.etiquetas = list(map(int, etiquetas))
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        enc = self.tok(
            self.textos[idx],
            truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.etiquetas[idx], dtype=torch.long)
        }


def _predict_probs(model, loader, device):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            p = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            probs.append(p)
    return np.concatenate(probs)


def entrenar_clasificador(modelo_nombre, patologia, X_train, y_train, X_test, y_test,
                          short_name, epochs, max_length=MAX_LENGTH,
                          batch_size=BATCH_SIZE, lr=LEARNING_RATE):
    """
    Fine-tunea un transformer binario para una patología.
    Si ya existe un checkpoint en models/<short_name>/<patologia>/, carga y solo evalúa.
    Retorna: dict con métricas {AUC, F1_pos, F1_neg, Kappa, threshold, n_train, n_pos_train}.
    """
    print(f"\n{'='*70}")
    print(f">>> {short_name.upper()}  |  {patologia.upper()}")
    print(f"{'='*70}")
    print(f"Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    out_dir = MODELS_DIR / short_name / patologia
    ya_entrenado = (out_dir / 'config.json').exists() and (out_dir / 'pytorch_model.bin').exists() or \
                   (out_dir / 'config.json').exists() and (out_dir / 'model.safetensors').exists()

    pos = int(np.sum(y_train == 1)); neg = int(np.sum(y_train == 0))
    print(f"Train: {neg} neg | {pos} pos ({pos/len(y_train)*100:.1f}%) | total {len(y_train)}")
    print(f"Test : {int(np.sum(y_test==0))} neg | {int(np.sum(y_test==1))} pos | total {len(y_test)}")

    # ---- Tokenizer ----
    tokenizer_src = str(out_dir) if ya_entrenado else modelo_nombre
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_src)

    test_ds = TextDataset(X_test, y_test, tokenizer, max_length)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    if ya_entrenado:
        print(f"Checkpoint encontrado en {out_dir} — se carga sin re-entrenar.")
        model = AutoModelForSequenceClassification.from_pretrained(str(out_dir)).to(DEVICE)
    else:
        out_dir.mkdir(parents=True, exist_ok=True)

        train_ds = TextDataset(X_train, y_train, tokenizer, max_length)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

        model = AutoModelForSequenceClassification.from_pretrained(
            modelo_nombre, num_labels=2
        ).to(DEVICE)

        # ---- class_weight para CrossEntropy ----
        clases = np.unique(y_train)
        if len(clases) < 2:
            cw = np.array([1.0, 1.0], dtype=np.float32)
        else:
            cw = compute_class_weight('balanced', classes=clases, y=y_train).astype(np.float32)
        cw_t = torch.tensor(cw, dtype=torch.float32, device=DEVICE)
        loss_fn = torch.nn.CrossEntropyLoss(weight=cw_t)
        print(f"class_weight: neg={cw[0]:.3f} pos={cw[1]:.3f}")

        # ---- Optimizador y scheduler ----
        optim = AdamW(model.parameters(), lr=lr)
        total_steps = len(train_loader) * epochs
        sched = get_linear_schedule_with_warmup(
            optim, num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps
        )

        # ---- Loop de entrenamiento ----
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                total_loss = 0.0
                pbar = tqdm(train_loader,
                            desc=f"  Ep {epoch}/{epochs} [{patologia}]",
                            leave=False)
                for batch in pbar:
                    optim.zero_grad()
                    input_ids = batch['input_ids'].to(DEVICE)
                    attention_mask = batch['attention_mask'].to(DEVICE)
                    labels = batch['labels'].to(DEVICE)

                    logits = model(input_ids=input_ids,
                                   attention_mask=attention_mask).logits
                    loss = loss_fn(logits, labels)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optim.step()
                    sched.step()

                    total_loss += loss.item()
                    pbar.set_postfix(loss=f"{loss.item():.4f}")

                avg_loss = total_loss / max(1, len(train_loader))
                print(f"  Época {epoch}/{epochs}  loss={avg_loss:.4f}")

            # ---- Guardar checkpoint ----
            model.save_pretrained(str(out_dir))
            tokenizer.save_pretrained(str(out_dir))
            print(f"Checkpoint guardado en {out_dir}")

        except Exception as e:
            print(f"ERROR durante entrenamiento de {patologia}: {e}")
            return {
                'AUC': np.nan, 'F1_pos': np.nan, 'F1_neg': np.nan,
                'Kappa': np.nan, 'threshold': np.nan,
                'n_train': len(y_train), 'n_pos_train': pos,
                'error': str(e)
            }

    # ---- Evaluación ----
    y_prob = _predict_probs(model, test_loader, DEVICE)

    # Umbral óptimo por F1 (igual que el LSTM)
    prec, rec, thr = precision_recall_curve(y_test, y_prob)
    f1_curve = 2 * prec * rec / (prec + rec + 1e-9)
    if len(thr) > 0 and np.any(~np.isnan(f1_curve[:-1])):
        best_thr = float(thr[int(np.nanargmax(f1_curve[:-1]))])
    else:
        best_thr = 0.5
    y_pred = (y_prob >= best_thr).astype(int)

    auc = roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) > 1 else np.nan
    cm = confusion_matrix(y_test, y_pred)
    f1_neg = 2*cm[0,0] / (2*cm[0,0] + cm[0,1] + cm[1,0]) if (2*cm[0,0] + cm[0,1] + cm[1,0]) > 0 else 0
    f1_pos = 2*cm[1,1] / (2*cm[1,1] + cm[0,1] + cm[1,0]) if (2*cm[1,1] + cm[0,1] + cm[1,0]) > 0 else 0
    kappa = cohen_kappa_score(y_test, y_pred)

    print(f"\nUmbral óptimo (F1): {best_thr:.3f}")
    print(classification_report(y_test, y_pred,
          target_names=ETIQUETAS.get(patologia, ['Negativo', 'Positivo']),
          digits=3, zero_division=0))
    print(f"AUC:    {auc:.4f}")
    print(f"Kappa:  {kappa:.4f}")
    print(f"F1-Pos: {f1_pos:.4f}  |  F1-Neg: {f1_neg:.4f}")
    print(f"Fin:    {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    # Liberar memoria
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'AUC': float(auc) if not np.isnan(auc) else np.nan,
        'F1_pos': float(f1_pos),
        'F1_neg': float(f1_neg),
        'Kappa': float(kappa),
        'threshold': float(best_thr),
        'n_train': int(len(y_train)),
        'n_pos_train': int(pos)
    }


def _datos_para_patologia(patologia):
    """Devuelve (X_train, y_train, X_test, y_test) según boost ROS si aplica."""
    if patologia in USA_ROS_BOOST:
        df_tr = df_train_acv
    else:
        df_tr = df_train
    X_tr = df_tr['texto'].astype(str).values
    y_tr = df_tr[patologia].astype(int).values
    X_te = df_test['texto'].astype(str).values
    y_te = df_test[patologia].astype(int).values
    return X_tr, y_tr, X_te, y_te


def entrenar_todas(modelo_clave):
    """Loop sobre las 4 patologías. Continúa aunque alguna falle."""
    cfg = MODELOS[modelo_clave]
    resultados = {}
    t0 = time.time()
    for pat in PATOLOGIAS:
        X_tr, y_tr, X_te, y_te = _datos_para_patologia(pat)
        try:
            resultados[pat] = entrenar_clasificador(
                modelo_nombre = cfg['hf_name'],
                patologia     = pat,
                X_train       = X_tr, y_train = y_tr,
                X_test        = X_te, y_test  = y_te,
                short_name    = cfg['short'],
                epochs        = cfg['epochs'],
            )
        except Exception as e:
            print(f"FALLÓ {pat}: {e} — se continúa con el resto.")
            resultados[pat] = {
                'AUC': np.nan, 'F1_pos': np.nan, 'F1_neg': np.nan,
                'Kappa': np.nan, 'threshold': np.nan,
                'n_train': int(len(y_tr)), 'n_pos_train': int(np.sum(y_tr==1)),
                'error': str(e)
            }
    elapsed = time.time() - t0
    print(f"\nTiempo total {modelo_clave}: {elapsed/60:.1f} min ({elapsed/3600:.2f} h)")
    return resultados, elapsed


def imprimir_resumen(nombre, resultados):
    print(f"\n{'='*72}")
    print(f"RESUMEN — {nombre}")
    print('='*72)
    print(f"{'Patología':<28} {'AUC':>8} {'F1-Pos':>10} {'F1-Neg':>10} {'Kappa':>10}")
    print('-'*72)
    aucs, kappas, f1pos = [], [], []
    for p in PATOLOGIAS:
        r = resultados.get(p, {})
        auc = r.get('AUC', np.nan); kp = r.get('Kappa', np.nan)
        fp  = r.get('F1_pos', np.nan); fn = r.get('F1_neg', np.nan)
        if not np.isnan(auc):   aucs.append(auc)
        if not np.isnan(kp):    kappas.append(kp)
        if not np.isnan(fp):    f1pos.append(fp)
        print(f"{p.capitalize():<28} {auc:>8.3f} {fp:>10.3f} {fn:>10.3f} {kp:>10.3f}")
    print('-'*72)
    if aucs:   print(f"{'AUC promedio':<28} {np.mean(aucs):>8.3f}")
    if f1pos:  print(f"{'F1-Pos promedio':<28} {' ':>8} {np.mean(f1pos):>10.3f}")
    if kappas: print(f"{'Kappa promedio':<28} {' ':>8} {' ':>10} {' ':>10} {np.mean(kappas):>10.3f}")
    print('='*72)

In [ ]:
# =============================================================================
# DistilBERT — DATOS CRÍTICOS (sin augmentación)
# Checkpoint en models/distilbert_criticos/ — no sobrescribe modelos LLM.
# =============================================================================

MODELOS['distilbert_criticos'] = {
    'hf_name': 'distilbert-base-multilingual-cased',
    'short':   'distilbert_criticos',
    'epochs':  3,
}

# Preservar globals LLM
_df_train_llm     = df_train
_df_test_llm      = df_test
_df_train_acv_llm = df_train_acv

# Apuntar globals al conjunto crítico (sin boost ROS para ACV)
df_train     = df_crit_train.copy()
df_test      = df_crit_test.copy()
df_train_acv = df_crit_train.copy()

print('=' * 70)
print('DistilBERT — DATOS CRÍTICOS (rad_criticos.xlsx)')
print(f'Train: {len(df_train)} | Test: {len(df_test)}')
print('=' * 70)
print(f'INICIO: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

resultados_criticos_distilbert, t_criticos = entrenar_todas('distilbert_criticos')

# Restaurar globals LLM
df_train     = _df_train_llm
df_test      = _df_test_llm
df_train_acv = _df_train_acv_llm
print('Globals restaurados al dataset LLM.')

imprimir_resumen('DistilBERT — Datos Críticos', resultados_criticos_distilbert)


## DistilBERT Multilingual

Más pequeño, ~6 capas. Idóneo para CPU. ~2-3 horas para 4 patologías × 3 épocas.

In [7]:
# Celda 4 — entrenamiento DistilBERT
print(f"INICIO DistilBERT: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
resultados_distilbert, t_distilbert = entrenar_todas('distilbert')

INICIO DistilBERT: 2026-04-29 21:24:57

>>> DISTILBERT  |  ACV
Inicio: 2026-04-29 21:24:57
Train: 1565 neg | 530 pos (25.3%) | total 2095
Test : 97 neg | 46 pos | total 143
class_weight: neg=0.669 pos=1.976


  Época 1/3  loss=0.3408


  Época 2/3  loss=0.1628


  Época 3/3  loss=0.0835
Checkpoint guardado en C:\Users\Isabella\Documents\ICESI\8. Octavo Semestre\Proyecto de Grado I\PDG-Diagnostico-Clinico\models\distilbert\acv

Umbral óptimo (F1): 0.024
              precision    recall  f1-score   support

     Sin ACV      0.940     0.969     0.954        97
         ACV      0.930     0.870     0.899        46

    accuracy                          0.937       143
   macro avg      0.935     0.919     0.927       143
weighted avg      0.937     0.937     0.936       143

AUC:    0.9756
Kappa:  0.8533
F1-Pos: 0.8989  |  F1-Neg: 0.9543
Fin:    2026-04-29 22:45:32

>>> DISTILBERT  |  HEMORRAGIA
Inicio: 2026-04-29 22:45:32
Train: 647 neg | 687 pos (51.5%) | total 1334
Test : 17 neg | 126 pos | total 143
class_weight: neg=1.031 pos=0.971


  Época 1/3  loss=0.4429


...

  Época 2/3  loss=0.2964


  Época 3/3  loss=0.2181
Checkpoint guardado en C:\Users\Isabella\Documents\ICESI\8. Octavo Semestre\Proyecto de Grado I\PDG-Diagnostico-Clinico\models\distilbert\desviacion_linea_media

Umbral óptimo (F1): 0.050
                               precision    recall  f1-score   support

Sin desviacion de linea media      0.946     0.868     0.905       121
    Desviacion de linea media      0.500     0.727     0.593        22

                     accuracy                          0.846       143
                    macro avg      0.723     0.798     0.749       143
                 weighted avg      0.877     0.846     0.857       143

AUC:    0.8952
Kappa:  0.5017
F1-Pos: 0.5926  |  F1-Neg: 0.9052
Fin:    2026-04-30 00:28:01

>>> DISTILBERT  |  FRACTURA_COMPLEJA_CRANEO
Inicio: 2026-04-30 00:28:01
Train: 1009 neg | 325 pos (24.4%) | total 1334
Test : 138 neg | 5 pos | total 143
class_weight: neg=0.661 pos=2.052


  Época 1/3  loss=0.3177


  Época 2/3  loss=0.1495


  Época 3/3  loss=0.0877
Checkpoint guardado en C:\Users\Isabella\Documents\ICESI\8. Octavo Semestre\Proyecto de Grado I\PDG-Diagnostico-Clinico\models\distilbert\fractura_compleja_craneo

Umbral óptimo (F1): 0.022
                                 precision    recall  f1-score   support

Sin fractura compleja de craneo      0.979     0.993     0.986       138
    Fractura compleja de craneo      0.667     0.400     0.500         5

                       accuracy                          0.972       143
                      macro avg      0.823     0.696     0.743       143
                   weighted avg      0.968     0.972     0.969       143

AUC:    0.9406
Kappa:  0.4865
F1-Pos: 0.5000  |  F1-Neg: 0.9856
Fin:    2026-04-30 01:18:31

Tiempo total distilbert: 233.6 min (3.89 h)


In [8]:
# Celda 5 — resumen DistilBERT
imprimir_resumen('DistilBERT Multilingual', resultados_distilbert)


RESUMEN — DistilBERT Multilingual
Patología                         AUC     F1-Pos     F1-Neg      Kappa
------------------------------------------------------------------------
Acv                             0.976      0.899      0.954      0.853
Hemorragia                      0.856      0.958      0.560      0.524
Desviacion_linea_media          0.895      0.593      0.905      0.502
Fractura_compleja_craneo        0.941      0.500      0.986      0.487
------------------------------------------------------------------------
AUC promedio                    0.917
F1-Pos promedio                            0.737
Kappa promedio                                                   0.591


## RoBERTa-BNE (PlanTL-GOB-ES)

Modelo en español entrenado por PlanTL/BNE. Más pesado que DistilBERT — ejecutar solo si hay tiempo. Esta sección es independiente y puede correrse sin re-entrenar DistilBERT.

In [9]:
# Celda 6 — entrenamiento RoBERTa-BNE
print(f"INICIO RoBERTa-BNE: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
resultados_roberta, t_roberta = entrenar_todas('roberta_bne')

INICIO RoBERTa-BNE: 2026-04-30 01:18:31

>>> ROBERTA_BNE  |  ACV
Inicio: 2026-04-30 01:18:31
Train: 1565 neg | 530 pos (25.3%) | total 2095
Test : 97 neg | 46 pos | total 143
FALLÓ acv: Can't load tokenizer for 'PlanTL-GOB-ES/roberta-base-bne'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'PlanTL-GOB-ES/roberta-base-bne' is the correct path to a directory containing all relevant files for a RobertaTokenizerFast tokenizer. — se continúa con el resto.

>>> ROBERTA_BNE  |  HEMORRAGIA
Inicio: 2026-04-30 01:18:33
Train: 647 neg | 687 pos (51.5%) | total 1334
Test : 17 neg | 126 pos | total 143
FALLÓ hemorragia: Can't load tokenizer for 'PlanTL-GOB-ES/roberta-base-bne'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'PlanTL-GOB-ES/roberta-base-bne' is the correct path to a directory 

In [10]:
# Celda 7 — resumen RoBERTa-BNE
imprimir_resumen('RoBERTa-BNE', resultados_roberta)


RESUMEN — RoBERTa-BNE
Patología                         AUC     F1-Pos     F1-Neg      Kappa
------------------------------------------------------------------------
Acv                               nan        nan        nan        nan
Hemorragia                        nan        nan        nan        nan
Desviacion_linea_media            nan        nan        nan        nan
Fractura_compleja_craneo          nan        nan        nan        nan
------------------------------------------------------------------------


## Comparación final: DistilBERT vs RoBERTa vs LSTM (LLM+ROS) vs SVM (LLM+ROS)

In [11]:
# Celda 8 — tabla comparativa final
# Baselines fijos (resultados ya reportados en notebooks 1 y 2)
BASELINE_LSTM_F1POS = {
    'acv':                       0.933,
    'hemorragia':                0.944,
    'desviacion_linea_media':    0.593,
    'fractura_compleja_craneo':  0.333,
}
BASELINE_LSTM_KAPPA_PROM = 0.480

BASELINE_SVM_F1POS = {
    'acv':                       0.966,
    'hemorragia':                0.948,
    'desviacion_linea_media':    0.927,
    'fractura_compleja_craneo':  0.750,
}
BASELINE_SVM_F1MACRO = 0.898


def _f1pos(resultados, p):
    return resultados.get(p, {}).get('F1_pos', np.nan)

filas = []
for p in PATOLOGIAS:
    filas.append({
        'Patología':       p.capitalize(),
        'SVM LLM+ROS':     BASELINE_SVM_F1POS[p],
        'LSTM LLM+ROS':    BASELINE_LSTM_F1POS[p],
        'DistilBERT':      _f1pos(resultados_distilbert, p) if 'resultados_distilbert' in globals() else np.nan,
        'RoBERTa-BNE':     _f1pos(resultados_roberta,    p) if 'resultados_roberta'    in globals() else np.nan,
    })

df_cmp = pd.DataFrame(filas).set_index('Patología')
print("\nF1-Pos por patología — comparativa")
print('='*72)
print(df_cmp.round(3).to_string())
print('='*72)

# Promedios (F1-Pos macro)
print("\nF1-Pos macro (promedio sobre las 4 patologías):")
for col in df_cmp.columns:
    vals = df_cmp[col].dropna().values
    if len(vals):
        print(f"  {col:<15s}  {np.mean(vals):.3f}")
    else:
        print(f"  {col:<15s}  (sin datos)")

# Kappa promedio cuando esté disponible
print("\nKappa promedio:")
print(f"  LSTM LLM+ROS    {BASELINE_LSTM_KAPPA_PROM:.3f}  (referencia notebook 2)")
if 'resultados_distilbert' in globals():
    kp = [resultados_distilbert[p].get('Kappa', np.nan) for p in PATOLOGIAS]
    kp = [k for k in kp if not np.isnan(k)]
    if kp:
        print(f"  DistilBERT      {np.mean(kp):.3f}")
if 'resultados_roberta' in globals():
    kp = [resultados_roberta[p].get('Kappa', np.nan) for p in PATOLOGIAS]
    kp = [k for k in kp if not np.isnan(k)]
    if kp:
        print(f"  RoBERTa-BNE     {np.mean(kp):.3f}")

print(f"\nReferencia SVM F1-macro reportado en notebook 2: {BASELINE_SVM_F1MACRO:.3f}")

# Tiempos totales
tot = 0.0
if 't_distilbert' in globals(): tot += t_distilbert
if 't_roberta'    in globals(): tot += t_roberta
if tot > 0:
    print(f"\nTiempo total de entrenamiento (transformers): {tot/60:.1f} min ({tot/3600:.2f} h)")


F1-Pos por patología — comparativa
                          SVM LLM+ROS  LSTM LLM+ROS  DistilBERT  RoBERTa-BNE
Patología                                                                   
Acv                             0.966         0.933       0.899          NaN
Hemorragia                      0.948         0.944       0.958          NaN
Desviacion_linea_media          0.927         0.593       0.593          NaN
Fractura_compleja_craneo        0.750         0.333       0.500          NaN

F1-Pos macro (promedio sobre las 4 patologías):
  SVM LLM+ROS      0.898
  LSTM LLM+ROS     0.701
  DistilBERT       0.737
  RoBERTa-BNE      (sin datos)

Kappa promedio:
  LSTM LLM+ROS    0.480  (referencia notebook 3)
  DistilBERT      0.591

Referencia SVM F1-macro reportado en notebook 2: 0.898

Tiempo total de entrenamiento (transformers): 233.7 min (3.89 h)


In [12]:
ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)
print("Modo normal restaurado")

Modo normal restaurado


## Comparación DistilBERT: Críticos vs LLM+ROS

AUC y F1-pos por patología comparando el fine-tuning sobre datos críticos originales versus datos aumentados con LLM+ROS.

In [ ]:
# =============================================================================
# COMPARACIÓN DistilBERT: CRÍTICOS vs LLM+ROS
# =============================================================================
print('\n' + '=' * 72)
print('COMPARACIÓN DistilBERT — CRÍTICOS vs LLM+ROS')
print('=' * 72)
print(f'{"Patología":<28} {"AUC Crit":>10} {"AUC LLM":>10} {"F1 Crit":>10} {"F1 LLM":>10}')
print('-' * 72)
for p in PATOLOGIAS:
    _rc = resultados_criticos_distilbert.get(p, {})
    _rl = resultados_distilbert.get(p, {})
    print(f'{p.capitalize():<28}'
          f' {_rc.get("AUC", float("nan")):>10.3f}'
          f' {_rl.get("AUC", float("nan")):>10.3f}'
          f' {_rc.get("F1_pos", float("nan")):>10.3f}'
          f' {_rl.get("F1_pos", float("nan")):>10.3f}')
print('-' * 72)
_prom_auc_c = np.nanmean([resultados_criticos_distilbert.get(p, {}).get('AUC', float('nan')) for p in PATOLOGIAS])
_prom_auc_l = np.nanmean([resultados_distilbert.get(p, {}).get('AUC', float('nan')) for p in PATOLOGIAS])
_prom_f1_c  = np.nanmean([resultados_criticos_distilbert.get(p, {}).get('F1_pos', float('nan')) for p in PATOLOGIAS])
_prom_f1_l  = np.nanmean([resultados_distilbert.get(p, {}).get('F1_pos', float('nan')) for p in PATOLOGIAS])
print(f'{"Promedio":<28} {_prom_auc_c:>10.3f} {_prom_auc_l:>10.3f} {_prom_f1_c:>10.3f} {_prom_f1_l:>10.3f}')
print('=' * 72)
